# F03 - NumPy and pandas for datasets

**Hardware:** CPU-only · **Time:** 2-3 hours · **Prerequisite:** F01

This notebook creates a synthetic student table, inspects it with pandas, and
uses NumPy to vectorize calculations. The table is deliberately small (400 rows)
so every operation is fast and the output is readable.

Expected shapes are printed and asserted. If an assertion fails, read the shape
before changing the code.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("numpy", np.__version__, "| pandas", pd.__version__)

numpy 2.5.4 | pandas 2.3.3


## 1. Build a table with a known shape

A DataFrame is a collection of columns with a shared index. Check `shape`,
`dtypes`, and the first rows before any analysis. The generator below uses a
logistic relationship for `passed`, so the data is realistic enough to model but
contains no real students.

In [2]:
def make_students(n: int = 400, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    study = rng.gamma(2.0, 2.0, n).clip(0, 20)
    attendance = rng.uniform(40, 100, n)
    prior = rng.normal(65, 12, n).clip(0, 100)
    sleep = rng.normal(7, 1.2, n).clip(3, 10)
    tutoring = rng.choice(["yes", "no"], n, p=[0.35, 0.65])
    course = rng.choice(["Math", "Physics", "CS"], n, p=[0.4, 0.3, 0.3])
    logit = (
        -11.5 + 0.30 * study + 0.055 * attendance + 0.075 * prior
        + 0.20 * sleep + 1.1 * (tutoring == "yes")
    )
    passed = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    final_score = (
        15 + 2.6 * study + 0.30 * attendance + 0.40 * prior
        + 0.8 * sleep + 4.0 * (tutoring == "yes") + rng.normal(0, 5, n)
    ).clip(0, 100)
    return pd.DataFrame(
        {
            "student_id": np.arange(1, n + 1),
            "study_hours": study.round(2),
            "attendance": attendance.round(1),
            "prior_score": prior.round(1),
            "sleep_hours": sleep.round(2),
            "tutoring": tutoring,
            "course": course,
            "passed": passed,
            "final_score": final_score.round(1),
        }
    )


df = make_students()
print("shape:", df.shape)
print(df.head(3).to_string(index=False))
assert df.shape == (400, 9)
assert df["student_id"].is_unique
assert set(df["course"]) == {"Math", "Physics", "CS"}

shape: (400, 9)
 student_id  study_hours  attendance  prior_score  sleep_hours tutoring  course  passed  final_score
          1         4.18        57.7         61.2         4.19       no      CS       0         84.4
          2         5.67        79.8         69.4         8.13      yes Physics       1         90.6
          3         3.67        90.2         72.4         5.82       no    Math       0         77.9


## 2. NumPy arrays: shape, dtype, and vectorization

`df["study_hours"].to_numpy()` returns a one-dimensional array of length 400.
Reshape it to `(400, 1)` when a model expects a two-dimensional feature matrix.
A vectorized mean is expressed in C and avoids a Python loop.

In [3]:
study = df["study_hours"].to_numpy()
print("shape:", study.shape, "dtype:", study.dtype)
assert study.shape == (400,)
assert study.ndim == 1

start = time.perf_counter()
loop_mean = sum(value for value in study) / len(study)
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
vector_mean = study.mean()
vector_seconds = time.perf_counter() - start

print("loop mean:", round(loop_mean, 4))
print("vector mean:", round(vector_mean, 4))
print("loop seconds:", round(loop_seconds, 6), "| vector seconds:", round(vector_seconds, 6))
assert np.isclose(loop_mean, vector_mean)

shape: (400,) dtype: float64
loop mean: 3.9994
vector mean: 3.9994
loop seconds: 8.8e-05 | vector seconds: 4.8e-05


## 3. Broadcasting and boolean masks

Broadcasting lets NumPy combine arrays with compatible trailing dimensions.
A boolean mask selects rows without a loop. The proportion of `True` values is
the pass rate.

In [4]:
scores = df["final_score"].to_numpy()
centered = scores - scores.mean()
print("centered shape:", centered.shape, "mean:", round(centered.mean(), 8))
assert centered.shape == (400,)
assert abs(centered.mean()) < 1e-9

passed_mask = df["passed"].to_numpy() == 1
print("pass rate:", round(passed_mask.mean(), 3))
assert passed_mask.sum() == int(df["passed"].sum())

passed_study = df.loc[passed_mask, "study_hours"]
failed_study = df.loc[~passed_mask, "study_hours"]
print("mean study hours | passed:", round(passed_study.mean(), 2))
print("mean study hours | failed:", round(failed_study.mean(), 2))
assert passed_study.mean() > failed_study.mean()

centered shape: (400,) mean: -0.0
pass rate: 0.507
mean study hours | passed: 4.73
mean study hours | failed: 3.25


## 4. Indexing with `loc` and `iloc`

- `loc` selects by label, including a boolean mask.
- `iloc` selects by integer position.
Mixing them is the most common pandas error. Use `loc` for named columns and
`iloc` for "first five rows".

In [5]:
print(df.loc[0, ["study_hours", "course", "passed"]].to_dict())
print(df.iloc[:3, :4].to_string(index=False))
assert df.loc[0, "student_id"] == 1
assert df.iloc[0]["student_id"] == df.loc[0, "student_id"]

{'study_hours': 4.18, 'course': 'CS', 'passed': 0}
 student_id  study_hours  attendance  prior_score
          1         4.18        57.7         61.2
          2         5.67        79.8         69.4
          3         3.67        90.2         72.4


## 5. Grouping and aggregation

`groupby` answers "one row per group" questions: average score by course, pass
rate by tutoring status, and count per group. Always check the group sizes; a
mean over two students is not comparable to a mean over 200.

In [6]:
by_course = (
    df.groupby("course", as_index=False)
    .agg(students=("student_id", "count"), mean_score=("final_score", "mean"), pass_rate=("passed", "mean"))
    .sort_values("mean_score", ascending=False)
)
print(by_course.round(3).to_string(index=False))
assert by_course.shape[0] == 3
assert by_course["students"].sum() == len(df)

by_tutoring = df.groupby("tutoring")["passed"].agg(["count", "mean"])
print(by_tutoring.round(3).to_string())

 course  students  mean_score  pass_rate
   Math       145      79.703      0.586
Physics       133      78.969      0.519
     CS       122      78.407      0.402
          count   mean
tutoring              
no          247  0.421
yes         153  0.647


## 6. Merge tables on a key

Real projects split information across tables. `merge` joins rows by a shared
key. Check the row count before and after: a many-to-many join can explode.

In [7]:
course_info = pd.DataFrame(
    {
        "course": ["Math", "Physics", "CS"],
        "department": ["Sciences", "Sciences", "Computing"],
        "credits": [3, 4, 3],
    }
)
merged = df.merge(course_info, on="course", how="left", validate="many_to_one")
print("merged shape:", merged.shape)
print(merged.groupby("department")["final_score"].mean().round(2).to_string())
assert merged.shape[0] == df.shape[0]
assert merged["credits"].notna().all()

merged shape: (400, 11)
department
Computing    78.41
Sciences     79.35


## 7. Prepare a numeric feature matrix

Standardization subtracts the mean and divides by the standard deviation. The
result has mean 0 and standard deviation 1 on the data used to fit it. In F05 and
F10 you will see why this must be fit on training data only.

In [8]:
features = df[["study_hours", "attendance", "prior_score", "sleep_hours"]]
X = features.to_numpy(dtype=float)
print("X shape:", X.shape)
assert X.shape == (400, 4)

X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)
print("column means:", np.round(X_scaled.mean(axis=0), 8))
print("column stds:", np.round(X_scaled.std(axis=0), 8))
assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-9)
assert np.allclose(X_scaled.std(axis=0), 1, atol=1e-9)

X shape: (400, 4)
column means: [ 0. -0. -0. -0.]
column stds: [1. 1. 1. 1.]


## 8. Save and reload a table

CSV is portable and readable but loses dtypes. Read the file back, compare
shapes, and verify a known value. The generated file is ignored by Git; it is
evidence for your lab, not a source file for the course.

In [9]:
csv_path = OUTPUT_DIR / "students.csv"
df.to_csv(csv_path, index=False)
reloaded = pd.read_csv(csv_path)
print("reloaded shape:", reloaded.shape)
assert reloaded.shape == df.shape
assert reloaded.loc[0, "student_id"] == df.loc[0, "student_id"]
print("wrote", csv_path)

reloaded shape: (400, 9)
wrote outputs\students.csv


## Check your work

- [ ] You can state the shape of a DataFrame and a feature matrix.
- [ ] You used a boolean mask without a Python loop.
- [ ] You can explain `loc` versus `iloc`.
- [ ] You verified a merge did not change the row count.
- [ ] `outputs/students.csv` exists.

**Next:** F04 - Data cleaning, missing values, categorical features, and normalization.